<a href="https://colab.research.google.com/github/xamrayevsobirjon/colab_loyihalarim/blob/main/PARALLEL_KOMPYUTER_ARXITEKTURASI_VA_DASTURLASH_(XAMRAYEV_SOBIRJON_SAPAROVICH)_Algoritmni_parallellashtirish_va_hisoblash_yadrosini_(Kernel)_optimallashtirish_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from numba import cuda, float32
import time

TILE_SIZE = 16

@cuda.jit
def tiled_gemm_kernel(A, B, C, N):
    s_A = cuda.shared.array(shape=(TILE_SIZE, TILE_SIZE), dtype=float32)
    s_B = cuda.shared.array(shape=(TILE_SIZE, TILE_SIZE), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    bx = cuda.blockIdx.x
    by = cuda.blockIdx.y

    row = by * TILE_SIZE + ty
    col = bx * TILE_SIZE + tx

    sum_val = float32(0.0)
    num_phases = int((N + TILE_SIZE - 1) / TILE_SIZE)

    for m in range(num_phases):
        if row < N and (m * TILE_SIZE + tx) < N:
            s_A[ty, tx] = A[row, m * TILE_SIZE + tx]
        else:
            s_A[ty, tx] = float32(0.0)

        if col < N and (m * TILE_SIZE + ty) < N:
            s_B[ty, tx] = B[m * TILE_SIZE + ty, col]
        else:
            s_B[ty, tx] = float32(0.0)

        cuda.syncthreads()

        for k in range(TILE_SIZE):
            sum_val += s_A[ty, k] * s_B[k, tx]

        cuda.syncthreads()

    if row < N and col < N:
        C[row, col] = sum_val

def run_ai_training_simulation():
    N = 4096
    print(f"[HISOB-KITOB] {N}x{N} o'lchamdagi matritsalar tayyorlanmoqda...")

    A = cuda.pinned_array((N, N), dtype=np.float32)
    B = cuda.pinned_array((N, N), dtype=np.float32)
    A[:] = np.random.rand(N, N).astype(np.float32)
    B[:] = np.random.rand(N, N).astype(np.float32)

    d_A = cuda.to_device(A)
    d_B = cuda.to_device(B)
    d_C = cuda.device_array((N, N), dtype=np.float32)

    threads_per_block = (TILE_SIZE, TILE_SIZE)
    blocks_per_grid_x = int(np.ceil(N / threads_per_block[0]))
    blocks_per_grid_y = int(np.ceil(N / threads_per_block[1]))
    blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

    print("[HISOB-KITOB] GPU Kernel (Shared Memory bilan) ishga tushirildi...")
    start_time = time.time()

    tiled_gemm_kernel[blocks_per_grid, threads_per_block](d_A, d_B, d_C, N)
    cuda.synchronize()

    end_time = time.time()

    h_C = d_C.copy_to_host()

    print(f"[NATIJA] Hisoblash muvaffaqiyatli yakunlandi!")
    print(f"[NATIJA] GPU da ishlash vaqti: {end_time - start_time:.4f} soniya.")

if __name__ == '__main__':
    run_ai_training_simulation()

[HISOB-KITOB] 4096x4096 o'lchamdagi matritsalar tayyorlanmoqda...
[HISOB-KITOB] GPU Kernel (Shared Memory bilan) ishga tushirildi...
[NATIJA] Hisoblash muvaffaqiyatli yakunlandi!
[NATIJA] GPU da ishlash vaqti: 2.5673 soniya.
